In [ ]:
from libraries import *
from parameters import *
from numpy import asarray
from numpy import savetxt
import matplotlib as mpl

%matplotlib inline

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
adata = sc.read(par_save_filename_8)
zs = ["K_0", "K_1","K_2", "K_3", "K_4", "K_5", "K_CONTROL"]

In [ ]:
adata.obs["subCellType"] = "DC2"
adata.obs.loc[adata.obs.leiden.isin(['3']), "subCellType"] = "MacDC"
adata.obs.loc[adata.obs.leiden.isin(['8']), "subCellType"] = "DC1"
adata.obs.loc[adata.obs.leiden.isin(['5']), "subCellType"] = "MReg"

In [ ]:
fBarMat = adata.obs[zs]
fBarMat["leiden"] = adata.obs['leiden']
fBarMat

In [ ]:
allGuidesPerSCT = pd.DataFrame()

for elem in zs:
    print(elem)
    k = pd.DataFrame(pd.crosstab(fBarMat[elem], fBarMat.leiden))
    k = k.loc[k.index == 1,]
    k["KOGuide"] = elem
    allGuidesPerSCT = allGuidesPerSCT.append(k)

allGuidesPerSCT['noOfKOGroupCells'] = allGuidesPerSCT.loc[:,["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]].sum(axis=1)

In [ ]:
for i in range(0,6):
    adata.obs[f'K_{i}'] = adata.obs[f'K_{i}'].astype(str)
    
adata.obs[f'K_CONTROL'] = adata.obs[f'K_CONTROL'].astype(str)

In [ ]:
sc.pl.umap(adata, color=zs, ncols=4, palette=["grey", "red"], groups="1")

In [ ]:
sc.tl.embedding_density(adata, groupby="K_0")
sc.tl.embedding_density(adata, groupby="K_1")
sc.tl.embedding_density(adata, groupby="K_2")
sc.tl.embedding_density(adata, groupby="K_3")
sc.tl.embedding_density(adata, groupby="K_4")
sc.tl.embedding_density(adata, groupby="K_5")
sc.tl.embedding_density(adata, groupby="K_CONTROL")

In [ ]:
for i in range(0,6):
    a = adata.obs[f'umap_density_K_{i}'].values
    b = adata.obs["umap_density_K_CONTROL"].values
    if(min(a - b) < 0):
        adata.obs[f'K_{i}_over_CONTROL'] = (a - b) 
    

In [ ]:
for i in range(0,6):
    sc.pl.umap(adata, color='K_'+str(i)+'_over_CONTROL', color_map='vlag' )

In [ ]:
for i in range(0,6):
    kk = sc.pl.embedding_density(adata, 
                                 key="umap_density_K_"+str(i), 
                                 group="1", 
                                 color_map="nipy_spectral", 
                                 bg_dotsize=15, 
                                 fg_dotsize=15,
                            title="K_"+str(i), return_fig=True)
    kk.set_size_inches(5.2,5)
    kk.savefig("umap_density_K_"+str(i)+".pdf")

In [ ]:
kk = sc.pl.embedding_density(adata, key="umap_density_K_CONTROL", 
                             group="1", 
                             bg_dotsize=15, 
                             fg_dotsize=15,
                        color_map="nipy_spectral",
                        title="CONTROL", return_fig=True)



kk.set_size_inches(5.2,5)

kk.savefig('umap_density_K_CONTROL.pdf')  